In [14]:
import pandas as pd
import re


df = pd.read_csv('08_investor_transactions (1).csv')

print(df.shape)
print(df.head())

(32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verifie

In [15]:
print("Unique values before cleanup:", df['transaction_type'].unique())

def clean_transaction_type(val):
    if pd.isna(val):
        return None
    val = str(val).strip().lower()
    val = re.sub(r'[^a-z]', '', val)
    if 'sip' in val:
        return 'SIP'
    elif 'lump' in val:
        return 'Lumpsum'
    elif 'redeem' in val:
        return 'Redemption'
    else:
        return 'Unknown'

df['transaction_type'] = df['transaction_type'].apply(clean_transaction_type)

print("Unique values after cleanup:", df['transaction_type'].unique())

Unique values before cleanup: ['SIP' 'Redemption' 'Lumpsum']
Unique values after cleanup: ['SIP' 'Unknown' 'Lumpsum']


In [16]:
invalid_amount = df[df['amount_inr'] <= 0]
print(f"Invalid amount rows: {len(invalid_amount)}")
print(invalid_amount.head())

df = df[df['amount_inr'] > 0].reset_index(drop=True)

Invalid amount rows: 0
Empty DataFrame
Columns: [investor_id, transaction_date, amfi_code, transaction_type, amount_inr, state, city, city_tier, age_group, gender, annual_income_lakh, payment_mode, kyc_status]
Index: []


In [17]:
print("Unique KYC statuses:", df['kyc_status'].unique())

df['kyc_status'] = df['kyc_status'].astype(str).str.strip().str.title()
valid_kyc = ['Verified', 'Pending', 'Rejected']
invalid_kyc = df[~df['kyc_status'].isin(valid_kyc)]
print(f"Rows with unexpected KYC values: {len(invalid_kyc)}")

Unique KYC statuses: ['Verified' 'Pending']
Rows with unexpected KYC values: 0


In [18]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce', dayfirst=False)

print("Unparseable dates:", df['transaction_date'].isna().sum())

Unparseable dates: 0


In [19]:
before = df.shape[0]
df = df.drop_duplicates()
print(f"Dropped {before - df.shape[0]} duplicate rows")

Dropped 0 duplicate rows


In [20]:
df.to_csv('C:/Users/deepu/desktop/Projects/Supply Chain Intelligence/clean_transactions.csv', index=False)
print("Saved clean_transactions.csv with", df.shape[0], "rows")

Saved clean_transactions.csv with 32778 rows
